# Estimating the incremental impact of retail promotions

**Data:** Rossmann daily store sales (Kaggle)  
**Question:** Do promotions move incremental sales, or are they mostly aligned with periods that would have been strong anyway?

This notebook walks through panel construction, identification choices, and three estimators: **two-way fixed effects** (store + ISO week), an **event-style** lead/lag specification, and **heterogeneity** (promo × store traits; DiD-style interactions).

---
## 1. Environment and imports

Run this notebook from the `notebooks/` folder **or** from the repo root. The next cell adds the project root to `sys.path` so `src/` imports resolve.

In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

from src.run_analysis import (
    add_promo_calendar_leads_lags,
    fit_panel,
    load_and_prepare,
)

print("Project root:", ROOT)

---
## 2. Load and describe the panel

- Keep **open** days with **positive** sales (standard for this competition’s target).
- Merge **store.csv** for competition distance and **Promo2** (recurring promo program).
- Build **log_sales**, **ISO week** (`%G-%V`), and **day-of-week**.

In [ ]:
df = load_and_prepare()

print("Rows:", len(df))
print("Stores:", df["Store"].nunique())
print("Date range:", df["Date"].min().date(), "→", df["Date"].max().date())
print("Share of promo days:", df["Promo"].mean().round(3))
df[["Sales", "Customers", "Promo", "log_sales"]].describe().round(2)

### 2a. Promo intensity over time

Exploratory plot: 7-day moving average of the daily promo indicator (national series—see next section).

In [ ]:
daily = df.groupby("Date", as_index=False)["Promo"].mean()
daily["ma7"] = daily["Promo"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["Date"], daily["ma7"], color="#2ca02c", lw=1.2)
ax.set_ylabel("Promo (7-day mean)")
ax.set_xlabel("Date")
ax.set_title("National promo calendar (smoothed)")
fig.tight_layout()
plt.show()

---
## 3. Identification: what the data allow

**Check:** On each calendar date, is `Promo` the same for every store? If yes, a **full calendar-date fixed effect** would absorb `Promo` and you cannot estimate a separate promo coefficient in that specification.

Here we instead use **store fixed effects** + **ISO-week fixed effects**, so identification comes from **within-store, within-week** variation (promo on vs off days in the same ISO week), plus day-of-week and holiday controls. That is a deliberate modeling choice—discuss limitations (e.g. selection of promo days *within* a week) in write-ups.

In [ ]:
by_date = df.groupby("Date")["Promo"].nunique()
assert (by_date <= 1).all()
print("Promo is constant across stores on each date (national schedule).")
print("Max distinct promo values per date:", by_date.max())

---
## 4. Main specification: two-way fixed effects

**Outcome:** `log_sales`  
**Treatment:** `Promo`  
**Fixed effects:** store (`EntityEffects`) + ISO week (`other_effects`)  
**Controls:** day-of-week, state holiday, school holiday  
**SE:** clustered by store

`fit_panel` in `src/run_analysis.py` wires `year_week` into `other_effects` for `PanelOLS`.

In [ ]:
formula_main = (
    "log_sales ~ Promo + C(dow) + C(StateHoliday) + SchoolHoliday + EntityEffects"
)
mod_main = fit_panel(df, formula_main)
res_main = mod_main.fit(cov_type="clustered", cluster_entity=True)

display(res_main.summary.tables[1])
print("Within R²:", round(res_main.rsquared_within, 4))
print("N obs:", int(res_main.nobs))

**Reading the promo coefficient:** Under a log outcome, $\exp(\hat\beta)-1$ is a rough proportional change in sales if the fixed-effects structure is taken as given. Interpret as associative unless you defend stronger assumptions.

---
## 5. Robustness: controlling for `log(Customers)`

Traffic can sit on the causal path from promo to sales. Adding `log(Customers)` is a **sensitivity check**—it may soak up part of the promo channel, so treat it as secondary to the main spec.

In [ ]:
formula_cust = (
    "log_sales ~ Promo + log(Customers) + C(dow) + C(StateHoliday)"
    " + SchoolHoliday + EntityEffects"
)
res_cust = fit_panel(df, formula_cust).fit(cov_type="clustered", cluster_entity=True)
promo_row = pd.DataFrame(
    {
        "spec": ["main", "+ log(Customers)"],
        "Promo_coef": [res_main.params["Promo"], res_cust.params["Promo"]],
        "Promo_SE": [res_main.std_errors["Promo"], res_cust.std_errors["Promo"]],
    }
)
display(promo_row)

---
## 6. Event study: leads and lags of the national promo calendar

We add **±3 day** leads/lags of the **national** promo indicator (same value for all stores on a date). This is a **diagnostic** for timing and persistence; coefficients are **not** the same estimand as the single `Promo` term in the main model because neighboring days partition overlapping variation.

In [ ]:
EV_K = 3
df_ev = add_promo_calendar_leads_lags(df, k=EV_K)
lead_cols = [f"promo_lead{i}" for i in range(1, EV_K + 1)]
lag_cols = [f"promo_lag{i}" for i in range(1, EV_K + 1)]
ev_formula = (
    "log_sales ~ "
    + " + ".join(lead_cols + ["Promo"] + lag_cols)
    + " + C(dow) + C(StateHoliday) + SchoolHoliday + EntityEffects"
)
res_ev = fit_panel(df_ev, ev_formula).fit(cov_type="clustered", cluster_entity=True)

rows = []
for i in range(1, EV_K + 1):
    c = f"promo_lead{i}"
    rows.append({"k": -i, "coef": res_ev.params[c], "se": res_ev.std_errors[c]})
rows.append({"k": 0, "coef": res_ev.params["Promo"], "se": res_ev.std_errors["Promo"]})
for i in range(1, EV_K + 1):
    c = f"promo_lag{i}"
    rows.append({"k": i, "coef": res_ev.params[c], "se": res_ev.std_errors[c]})
ev_coef = pd.DataFrame(rows).sort_values("k")
display(ev_coef.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(0, color="0.4", lw=1)
ax.errorbar(
    ev_coef["k"],
    ev_coef["coef"],
    yerr=1.96 * ev_coef["se"],
    fmt="o",
    capsize=3,
    color="#1f77b4",
)
ax.set_xlabel("k (negative = lead; 0 = today)")
ax.set_ylabel("Coefficient (log sales)")
ax.set_title("Event-style leads/lags (same FE as main spec)")
fig.tight_layout()
plt.show()

---
## 7. Heterogeneity: DiD-style interactions

Time-invariant store traits (`high_comp`, `Promo2`) cannot enter **with a main effect** alongside **store fixed effects**—they are absorbed. We can still estimate **interactions** `Promo × high_comp` and `Promo × Promo2`:

- **high_comp:** competition distance above the cross-sectional median  
- **Promo2:** store participates in the recurring promo program (from `store.csv`)

In [ ]:
formula_het = (
    "log_sales ~ Promo + Promo:high_comp + Promo:Promo2 + C(dow) + C(StateHoliday)"
    " + SchoolHoliday + EntityEffects"
)
res_het = fit_panel(df, formula_het).fit(cov_type="clustered", cluster_entity=True)

cols = ["Promo", "Promo:high_comp", "Promo:Promo2"]
het_tbl = pd.DataFrame(
    {
        "coef": res_het.params[cols].values,
        "se": res_het.std_errors[cols].values,
        "p": res_het.pvalues[cols].values,
    },
    index=cols,
)
display(het_tbl.round(4))

labels = [
    "Promo (baseline; low-competition ref)",
    "+ Promo × high competition",
    "+ Promo × Promo2 store",
]
fig, ax = plt.subplots(figsize=(7, 3.5))
y = np.arange(len(labels))
coefs = [res_het.params[c] for c in cols]
ses = [res_het.std_errors[c] for c in cols]
ax.axvline(0, color="0.5", lw=1)
ax.errorbar(coefs, y, xerr=[1.96 * s for s in ses], fmt="o", capsize=4, color="#d62728")
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("Coefficient (log sales)")
fig.tight_layout()
plt.show()

---
## 8. Takeaways (short)

1. **National promo flag** implies you cannot combine a **date FE** and a **standalone** promo term; week FE + store FE is the main workaround used here.
2. **Main FE** results show a large within-sample association of promo days with higher log sales under the stated controls.
3. **Leads/lags** are useful diagnostics, not a second copy of the main estimand.
4. **Interactions** describe how the promo association differs for Promo2 stores vs others (and high vs low competition), not a second causal estimand without further assumptions.
